# Aggregate climate data and export to Chap CSV

This notebook shows how to connect to a running instance of Open Climate Service (OCS), aggregate climate data to administrative boundaries, and export the results to a CSV file compatible with the [Chap Modeling Platform](https://chap.dhis2.org/).

## Prerequisites

- A running instance of OCS, containing:
    1. An imported raster dataset of a climate variable.
    2. An imported vector dataset of administrative areas to aggregate to.

## Imports

In [1]:
import geopandas as gpd
import xarray as xr
import rioxarray
import pandas as pd
import openeo

## Connect to OCS

If you don't already have an OCS instance, see the [installation guide](https://open-climate-service.dhis2.org/setup_guide/) to get one up and running.

OCS instances are tied to particular countries and bounding extents. In this notebook we showcase the results for a Rwanda OCS instance, but the notebook will work for whichever country or area your OCS instance is configured for. 

Provide your instance url below and confirm that you can connect to it:

In [2]:
url = "http://localhost:9000"
conn = openeo.connect(url)
conn

<Connection to 'http://localhost:9000' with NullAuth>


## Climate data

Make sure you have imported a climate dataset, containing the data which you would like to aggregate. For this notebook we showcase the aggregation of ERA5-Land temperature, but you can also specify another dataset id below. 

In [9]:
# imported dataset id
climate_dataset_id = "era5land_temperature_daily"

# connect to the dataset
climate = conn.load_collection(
    climate_dataset_id,
)

# download to disk
download_to = "outputs/climate.nc"
climate.download(
    download_to,
    format="NETCDF",
)

# open with xarray to explore
ds = xr.open_dataset(download_to)
ds["band_data"].isel(band=0).plot()

ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

## Administrative Areas

Make sure you have imported the administrative areas that you want to use for aggregating and reporting climate statistics. These are the locations for which climate data is reported in your final Chap CSV file. 

You can use any polygon-based vector dataset, but an easy way to get started is using the builtin Overture Maps vector data source:

- Configure which boundaries you would like to import following [this guide](https://open-climate-service.dhis2.org/built_in_datasets/?h=overture#overture-maps-administrative-divisions-feature-collection).
- Trigger the import from the [Overture Maps data source page](http://localhost:9000/data-sources/overture_divisions). 

In [10]:
# the dataset id for imported boundaries
boundaries_dataset_id = "overture_divisions"

print(conn.list_collection_ids())
fdasfa

# inspect the data
boundaries = conn.load_collection(boundaries_dataset_id)
boundaries

['cop_elevation_static', 'era5heat_utci_daily', 'era5heat_utci_daily_max', 'era5land_temperature_daily', 'worldpop_agesex_global2_100m']


NameError: name 'fdasfa' is not defined

## Aggregate to Chap CSV

All instances of OCS comes with a builtin workflow for spatially aggregating data to Chap-compatible CSV format - which we run below:

In [ ]:
csvdata = conn.download(
    {
        "process_graph": {
            "1": {
                "process_id": "aggregate_to_chap_csv",
                "arguments": {
                    "dataset_id": CLIMATE_DATASET_ID,
                    "temporal_extent": ["2025-01-01", "2025-01-31"],
                    "geometries": gdf.__geo_interface__,
                    "method": "mean",
                    "period_type": "day",
                },
                "result": True,
            }
        }
    }
)

## Save CSV to file

The workflow returns the byte data for the resulting CSV file, which we then save to disk:

In [ ]:
# save to disk
csv_path = "outputs/aggregate_chap.csv"
with open(csv_path, "wb") as w:
    w.write(csvdata)

## Inspect the results

In [ ]:
# load the csv file to pandas
df = pd.read_csv(csv_path)

# inspect the data
print(df[df.time_period == 20250101])